# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np

from bodaqs_analysis import run_macro


In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\Simple tests\2025-12-21_11-57-23.CSV"   # change to your log
SCHEMA_PATH = r"Jupyter Notebook\event_schema.yaml"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock [mm]": 170.0,
    "rear_shock [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# Run the macro pipeline
results = run_macro(
    CSV_PATH,
    SCHEMA_PATH,
    normalize_ranges=NORMALIZE_RANGES,
    sample_rate_hz=SAMPLE_RATE_HZ,
)
session = results['session']
data = session['df']
events_df = results['events']
metrics_df = results['metrics']

data.head()


[Detect] Warning: invalid dt; skipping time-based distances/edge windows; using prominence-only.


AttributeError: 'tuple' object has no attribute 'get'

In [ ]:
data2, norm_report = normalize_and_scale(
    data,
    NORMALIZE_RANGES,
    zeroing_enabled=True,
    zero_window_s=1.0,
    clip_0_1=False,
    add_zeroed_column=True,
)
pd.DataFrame(norm_report)


In [ ]:
# Use the *_zeroed columns for VA, but output *_vel / *_acc on the base column names
zeroed_cols = [c + "_zeroed" for c in NORMALIZE_RANGES.keys() if (c + "_zeroed") in data2.columns]

va_df, va_meta = estimate_va_from_zeroed(
    data2,
    cols=zeroed_cols,
    sample_rate_hz=SAMPLE_RATE_HZ,
    time_col="t" if "t" in data2.columns else ("time_s" if "time_s" in data2.columns else None),
    window_points=31,
    poly_order=3,
    vel_suffix="_vel",
    acc_suffix="_acc",
    strip_zeroed_suffix=True,
)
va_df.filter(regex="(^t$|time|_vel$|_acc$)").head()


In [ ]:
schema, schema_hash = load_event_schema(SCHEMA_PATH)

events_df = detect_events_from_schema(va_df, schema)
from bodaqs_analysis import extract_metrics_df
metrics_df = extract_metrics_df(events_df)

events_df.head(), metrics_df.head()


In [ ]:
# Quick aggregate view (example): histogram of one metric if present
import matplotlib.pyplot as plt

col = next((c for c in metrics_df.columns if c.lower().endswith("peak_vel") or "peak" in c.lower()), None)
if col is None:
    print("No obvious peak metric column found. Available:", list(metrics_df.columns)[:25])
else:
    metrics_df[col].dropna().plot(kind="hist", bins=40)
    plt.title(col)
    plt.show()
